In [ ]:
# ============================================================
# CELL 1: Install Libraries
# ============================================================
# hcipy   → High Contrast Imaging for Python
#           Used to simulate the atmosphere + SH-WFS optically
#           This is how we generate training data
#
# aotools  → Adaptive Optics tools
#           Zernike polynomials, r0 estimation, turbulence stats
#           Directly implements formulas from Noll (1976)
#
# opencv   → For BMP reading and centroiding
#           C++ backend = fast (satisfies ISRO speed requirement)
#
# astropy  → For FITS files if ISRO provides them later
# scipy    → Signal processing: cross-correlation for τ₀
# ============================================================

!pip install hcipy aotools opencv-python-headless astropy scipy matplotlib numpy --quiet

print(" All libraries installed successfully")
print("These are the open-source libraries ISRO referenced in their problem statement.")

---
##  CELL 2 — Import everything + Mount Google Drive
**Why Drive?** Colab resets after 12 hours. Every output must be saved to Drive immediately.

In [ ]:


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2
import os
import time
import warnings
warnings.filterwarnings('ignore')

# HCIPy — our atmosphere + optical simulator
import hcipy

# AOtools — our AO math toolkit
import aotools

# Mount Google Drive — CRITICAL: save everything here
from google.colab import drive
drive.mount('/content/drive')

# Create our project folder structure on Drive
PROJECT_DIR = '/content/drive/MyDrive/PS09_BAH2026'
DAY1_DIR    = f'{PROJECT_DIR}/Day1_Outputs'
DATA_DIR    = f'{PROJECT_DIR}/Dataset'
MODEL_DIR   = f'{PROJECT_DIR}/Models'

for d in [PROJECT_DIR, DAY1_DIR, DATA_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print(f" Libraries imported | HCIPy v{hcipy.__version__}")
print(f" Drive mounted")
print(f" Project folders created at: {PROJECT_DIR}")
print()
print("Folder structure:")
print("  PS09_BAH2026/")
print("  ├── Day1_Outputs/   ← today's simulation images")
print("  ├── Dataset/        ← training data (Day 4)")
print("  └── Models/         ← trained CNN checkpoints (Day 4)")

In [ ]:


# --- Telescope / Beam Parameters ---
#  "Pupil size of the turbulated beam"
D = 2.0                   # Telescope/beam diameter in metres
                           # (lab setup will have a smaller beam, e.g., 10mm
                           #  but physics scales — we use 2m for IAO context)

# --- Wavelength ---
WAVELENGTH = 500e-9        # 500 nanometres = green light
                           # Typical for SH-WFS guide star systems

# --- MLA (Microlens Array) Parameters ---
# ISRO provides: "Basic information of MLA: size, number of lenslets, focal length"
N_LENSLETS  = 10           # Number of lenslets per side (10x10 = 100 subapertures)
                           # Real systems: 20x20 to 40x40
                           # We start with 10x10 for computational speed on Colab CPU
MLA_FOCAL_LENGTH = 5e-3    # 5mm focal length per lenslet

# --- Camera Parameters ---
#  "pixel size and frame resolution"
PIXEL_SIZE  = 10e-6        # 10 micrometres per pixel (typical science camera)
GRID_SIZE   = 128          # Simulation grid: 128x128 pixels
                           # (Real camera: 512x512 or 1024x1024)

# --- Turbulence Levels to Simulate ---
# These 3 values represent: good, moderate, and strong turbulence
# Your CNN must handle ALL of them — especially r0=0.03 (where classics fail)
R0_VALUES = {
    'good'     : 0.20,   # r₀ = 20cm — easy seeing, classics work fine
    'moderate' : 0.10,   # r₀ = 10cm — typical Indian observatory night
    'strong'   : 0.03,   # r₀ = 3cm  — strong turbulence, classics FAIL here
}

# --- Derived parameters ---
N_SUBAPERTURES = N_LENSLETS ** 2   # Total number of lenslets
N_SLOPES       = N_SUBAPERTURES * 2  # Each subaperture gives (sx, sy)

print("=" * 55)
print("PS09 BAH 2026 — Simulation Parameters")
print("=" * 55)
print(f"Telescope diameter     : {D}m")
print(f"Wavelength             : {WAVELENGTH*1e9:.0f}nm")
print(f"Lenslets per side      : {N_LENSLETS}")
print(f"Total subapertures     : {N_SUBAPERTURES}")
print(f"Slope vector length    : {N_SLOPES} values per frame")
print(f"MLA focal length       : {MLA_FOCAL_LENGTH*1e3:.0f}mm")
print(f"Pixel size             : {PIXEL_SIZE*1e6:.0f}µm")
print(f"Grid size              : {GRID_SIZE}x{GRID_SIZE}")
print(f"Turbulence levels      : {list(R0_VALUES.keys())}")
print("=" * 55)
print()
print("ISRO will provide real values for all these parameters.")
print("   Your code accepts them as inputs — never hardcode!")

In [ ]:
# ============================================================
# CELL 4: Simulate Wavefront Phase Maps
# Physics: atmosphere creates phase distortions φ(x,y)
# Output: 3 phase maps (one per turbulence level)
# ============================================================

def simulate_wavefront_phase(r0, D=2.0, wavelength=500e-9,
                              grid_size=128, wind_speed=10.0,
                              seed=None):
   
    if seed is not None:
        np.random.seed(seed)

    # Step 1: Create the pupil grid
    # This is the 2D grid representing the telescope aperture
    # Physics: the aperture is what limits the telescope's resolution
    pupil_grid = hcipy.make_pupil_grid(grid_size, D)

    # Step 2: Create circular aperture mask
    # Only the circular area inside the telescope tube collects light
    aperture = hcipy.circular_aperture(D)(pupil_grid)

    # Step 3: Compute Cn² from r₀
    # Cn² is the refractive index structure constant — how "lumpy" the air is
    # r₀ and Cn² are two ways to express the same turbulence strength
    Cn_squared = hcipy.Cn_squared_from_fried_parameter(r0, wavelength)

    # Step 4: Create atmospheric turbulence layer
    # Kolmogorov turbulence with given Cn², outer scale 100m, wind [v,0] m/s
    # Physics: one thin layer approximation (real atmosphere has many layers)
    layer = hcipy.InfiniteAtmosphericLayer(
        pupil_grid,
        Cn_squared,
        100,              # Outer scale L₀ in metres (largest eddy size)
        [wind_speed, 0],  # Wind velocity vector [vx, vy] in m/s
        1                 # Number of sub-harmonics (for low-frequency accuracy)
    )

    # Step 5: Create a flat wavefront (what we'd have WITHOUT atmosphere)
    # Physics: flat wavefront = plane-parallel = what comes from a distant star
    wf_flat = hcipy.Wavefront(aperture, wavelength)

    # Step 6: Propagate flat wavefront THROUGH the turbulent atmosphere
    # This is the distortion happening — the flat wave becomes crinkled
    wf_turbulent = layer.forward(wf_flat)

    # Step 7: Extract the phase map
    # .phase gives us φ(x,y) in radians
    # Convert to nanometres: φ_nm = φ_rad × (λ / 2π)
    phase_radians = wf_turbulent.phase.shaped  # 2D numpy array
    phase_nm      = phase_radians * (wavelength * 1e9) / (2 * np.pi)

    # Mask outside the aperture (set to NaN so it doesn't affect statistics)
    aperture_mask  = aperture.shaped > 0.5
    phase_nm_masked = phase_nm.copy()
    phase_nm_masked[~aperture_mask] = np.nan

    return phase_nm_masked, wf_turbulent, pupil_grid, aperture


# ── Simulate wavefronts for all 3 turbulence levels ──
print("Simulating wavefront phase maps...")
print("(Each simulation = what atmosphere does to starlight at that turbulence level)")
print()

wavefronts = {}
for label, r0 in R0_VALUES.items():
    t_start = time.time()
    phase, wf, grid, aperture = simulate_wavefront_phase(
        r0, D=D, wavelength=WAVELENGTH,
        grid_size=GRID_SIZE, wind_speed=10.0, seed=42
    )
    elapsed = time.time() - t_start

    # Compute RMS wavefront error (ignoring NaN pixels outside aperture)
    rms_nm = np.sqrt(np.nanmean(phase**2))

    wavefronts[label] = {
        'r0'       : r0,
        'phase_nm' : phase,
        'wf_obj'   : wf,
        'grid'     : grid,
        'aperture' : aperture,
        'rms_nm'   : rms_nm
    }

    # Strehl ratio (how sharp the image is after turbulence)
    # Marechal approximation: Strehl = exp(-(2π σ/λ)²)
    sigma_rad = (2 * np.pi * rms_nm) / (WAVELENGTH * 1e9)
    strehl    = np.exp(-sigma_rad**2)

    print(f"r₀={r0*100:.0f}cm ({label:8s}) | "
          f"RMS = {rms_nm:7.1f}nm | "
          f"Strehl = {strehl:.3f} | "
          f"Time: {elapsed:.2f}s")

print()
print(" RMS: how distorted the wavefront is (lower = better)")
print(" Strehl: image sharpness 0-1 (1.0 = perfect, < 0.2 = very blurry)")
print(" YOUR CNN must improve Strehl from current values toward 1.0")

In [ ]:
# ============================================================
# CELL 5: Visualise Wavefront Phase Maps
# Output: day1_wavefronts_comparison.png → saved to Drive
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Wavefront Phase Maps φ(x,y) — The CNN\'s Reconstruction Target\n'
    'Red = wavefront AHEAD of flat | Blue = wavefront BEHIND flat',
    fontsize=13, fontweight='bold', y=1.02
)

colors = ['good', 'moderate', 'strong']
color_map = 'RdBu_r'

for idx, (label, data) in enumerate(wavefronts.items()):
    phase = data['phase_nm']
    r0    = data['r0']
    rms   = data['rms_nm']

    # Symmetric color scale around zero
    vmax = np.nanpercentile(np.abs(phase), 98)

    im = axes[idx].imshow(
        phase, cmap=color_map,
        vmin=-vmax, vmax=vmax,
        origin='lower'
    )

    # Title with physics interpretation
    sigma_rad = (2 * np.pi * rms) / (WAVELENGTH * 1e9)
    strehl    = np.exp(-sigma_rad**2)

    axes[idx].set_title(
        f'r₀ = {r0*100:.0f}cm ({label} turbulence)\n'
        f'RMS = {rms:.1f}nm | Strehl = {strehl:.3f}',
        fontsize=11, pad=10
    )
    axes[idx].set_xlabel('x (pixels)', fontsize=10)
    axes[idx].set_ylabel('y (pixels)', fontsize=10)

    plt.colorbar(im, ax=axes[idx], label='Phase (nm)', fraction=0.046, pad=0.04)

    # Add annotation explaining what team is looking at
    if label == 'good':
        axes[idx].text(5, 5, 'Smooth gradients\nClassics work here',
                      color='white', fontsize=8, va='bottom',
                      bbox=dict(boxstyle='round', facecolor='green', alpha=0.7))
    elif label == 'moderate':
        axes[idx].text(5, 5, 'More variation\nClassics partially work',
                      color='white', fontsize=8, va='bottom',
                      bbox=dict(boxstyle='round', facecolor='orange', alpha=0.7))
    else:
        axes[idx].text(5, 5, 'Chaotic + branch points\nCNN required here!',
                      color='white', fontsize=8, va='bottom',
                      bbox=dict(boxstyle='round', facecolor='red', alpha=0.7))

plt.tight_layout()

# Save to Drive — this image goes in your proposal
save_path = f'{DAY1_DIR}/day1_wavefronts_comparison.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight')
plt.show()

print(f" Saved: {save_path}")
print()
print(" What to notice in this image:")
print("  r₀=20cm: smooth, large blobs of colour → easy to reconstruct")
print("  r₀=10cm: more variation, smaller features → moderate difficulty")
print("  r₀=3cm : chaotic, rapid changes → traditional algorithms FAIL here")
print("           Notice small isolated regions = branch points")
print()
print(" Your CNN's job: given a spot IMAGE, predict an array like these.")
print("   This is your reconstruction target W(xi, yi).")

In [ ]:
# ============================================================
# CELL 6: Generate SH-WFS Spot Pattern Images
# Physics: wavefront → MLA → camera → spot grid
# Output: spot images = what ISRO's camera records as BMP files
# ============================================================

def simulate_shwfs_frame(wf_turbulent, pupil_grid, aperture,
                          n_lenslets=10, mla_focal_length=5e-3,
                          D=2.0, add_noise=False, noise_level=0.01):
    

   
    lenslet_pitch = D / n_lenslets  # Physical size of each lenslet

    shwfs_optics = hcipy.SquareShackHartmannWavefrontSensorOptics(
        pupil_grid.scaled(mla_focal_length / D),  # scale grid to focal length
        mla_focal_length,
        n_lenslets,
        mla_focal_length
    )

    # Create slope estimator
    # Physics: this finds the centroid of each spot and computes slope
    shwfs_estimator = hcipy.ShackHartmannWavefrontSensorEstimator(
        shwfs_optics.mla_grid,
        shwfs_optics.micro_lens_array.mla_index
    )

    # Create a noiseless detector
    # In real setup: ISRO's science-grade camera records BMP files
    camera = hcipy.NoiselessDetector()

    # Propagate turbulent wavefront through MLA to camera
    # Physics: each lenslet creates one focused spot on the camera
    camera.integrate(shwfs_optics.forward(wf_turbulent), 1)
    raw_image = camera.read_out()

    # Get the 2D spot image (this is your BMP file equivalent)
    spot_image = raw_image.shaped.astype(np.float32)

    # Normalise to [0, 255] for BMP compatibility
    spot_image_norm = spot_image - spot_image.min()
    if spot_image_norm.max() > 0:
        spot_image_norm = (spot_image_norm / spot_image_norm.max() * 255).astype(np.uint8)

    # Add optional readout noise (simulates real camera)
    if add_noise:
        noise = np.random.normal(0, noise_level * 255, spot_image_norm.shape)
        spot_image_norm = np.clip(spot_image_norm.astype(float) + noise, 0, 255).astype(np.uint8)

    # Extract slope measurements from spot pattern
    # Physics: centroid displacement → wavefront slope at each subaperture
    slopes = shwfs_estimator.estimate([raw_image])

    camera.clear()

    return spot_image_norm, slopes, shwfs_optics, shwfs_estimator


# ── Generate spot images for all 3 turbulence levels ──
print("Generating SH-WFS spot pattern images...")
print("(This is what ISRO's camera records as BMP files)")
print()

spot_data = {}
for label, data in wavefronts.items():
    t_start = time.time()

    spot_img, slopes, shwfs_obj, shwfs_est = simulate_shwfs_frame(
        data['wf_obj'],
        data['grid'],
        data['aperture'],
        n_lenslets=N_LENSLETS,
        mla_focal_length=MLA_FOCAL_LENGTH,
        D=D
    )

    elapsed = time.time() - t_start

    spot_data[label] = {
        'spot_image'    : spot_img,
        'slopes'        : slopes,
        'shwfs_obj'     : shwfs_obj,
        'shwfs_est'     : shwfs_est,
    }

    print(f"r₀={data['r0']*100:.0f}cm ({label:8s}) | "
          f"Spot image shape: {spot_img.shape} | "
          f"Slopes shape: {np.array(slopes).shape} | "
          f"Time: {elapsed:.2f}s")

print()
print(f" Slopes array has {N_SLOPES} values = {N_SUBAPERTURES} subapertures × 2 (x and y)")
print(" These slopes are the INPUT to classical reconstruction algorithms")
print(" The spot IMAGE is the additional input to ISNet (dual-input CNN)")

---
##  CELL 7 — Visualise CNN Input vs CNN Target
**The most important visualisation of Day 1.** Top row = what the CNN sees (spot images). Bottom row = what the CNN must predict (phase maps). This is your entire problem in one figure.

In [ ]:
# ============================================================
# CELL 7: CNN Input (Spots) vs CNN Target (Phase) Side by Side
# This is YOUR PROBLEM visualised completely
# ============================================================

fig = plt.figure(figsize=(20, 10))
gs  = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.3)

fig.suptitle(
    'PS09 Problem Visualised: CNN Input (top) → CNN Target (bottom)\n'
    'Given the spot image, reconstruct the phase map — for all turbulence levels',
    fontsize=13, fontweight='bold'
)

for col, (label, r0) in enumerate(R0_VALUES.items()):
    spot_img  = spot_data[label]['spot_image']
    phase_map = wavefronts[label]['phase_nm']
    rms       = wavefronts[label]['rms_nm']

    # Top row: SH-WFS spot image (CNN input)
    ax_top = fig.add_subplot(gs[0, col])
    ax_top.imshow(spot_img, cmap='hot', origin='lower')
    ax_top.set_title(
        f'CNN INPUT — r₀={r0*100:.0f}cm ({label})\n'
        f'SH-WFS Spot Pattern (BMP file)\n'
        f'Shape: {spot_img.shape}',
        fontsize=10
    )
    ax_top.set_xlabel('Pixel x')
    ax_top.set_ylabel('Pixel y')

    # Add arrow showing spot displacement at a few locations
    if label != 'good':
        ax_top.text(5, spot_img.shape[0]-15,
                   '← spots shift\n   with wavefront slope',
                   color='cyan', fontsize=7,
                   bbox=dict(boxstyle='round', facecolor='black', alpha=0.6))

    # Bottom row: wavefront phase map (CNN target)
    ax_bot = fig.add_subplot(gs[1, col])
    vmax    = np.nanpercentile(np.abs(phase_map), 98)
    im      = ax_bot.imshow(phase_map, cmap='RdBu_r',
                             vmin=-vmax, vmax=vmax, origin='lower')
    ax_bot.set_title(
        f'CNN TARGET — W(xi,yi)\n'
        f'Wavefront Phase Map (nm)\n'
        f'RMS = {rms:.1f}nm',
        fontsize=10
    )
    ax_bot.set_xlabel('Pixel x')
    ax_bot.set_ylabel('Pixel y')
    plt.colorbar(im, ax=ax_bot, label='Phase (nm)', fraction=0.046, pad=0.04)

    # Draw arrow between rows
    fig.text(
        0.17 + col * 0.33, 0.51,
        '↓ CNN must\npredict this\nfrom above ↑',
        ha='center', va='center', fontsize=9,
        color='#534AB7', fontweight='bold'
    )

# Save
save_path = f'{DAY1_DIR}/day1_cnn_input_vs_target.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight')
plt.show()

print(f" Saved: {save_path}")
print()
print("Key observations for your team:")
print("  1. At r₀=20cm: spots are in a neat regular grid → phase is smooth")
print("  2. At r₀=3cm : spots are irregular, some merged → phase is chaotic")
print("  3. The CNN must learn this MAPPING: spot image → phase map")
print("  4. This is image-to-image translation — exactly like pix2pix!")
print()
print(" This figure will go in your BAH 2026 proposal as Figure 1.")

---
##  CELL 8 — Understand Slope Vectors
**Physics connection (Topic 3):** The slopes are what classical algorithms use for reconstruction. They are also the SECOND input to ISNet (dual-input CNN). Understanding what a slope vector looks like physically is critical.

In [ ]:
# ============================================================
# CELL 8: Visualise Slope Vectors
# Physics: slope_x[i] = how much the wavefront tilts in x at subaperture i
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    'SH-WFS Slope Vectors — Wavefront Gradients at Each Subaperture\n'
    'Top: X-slopes | Bottom: Y-slopes | These are the classical reconstruction inputs',
    fontsize=12, fontweight='bold'
)

for col, (label, r0) in enumerate(R0_VALUES.items()):
    slopes     = np.array(spot_data[label]['slopes']).flatten()
    n_sub      = len(slopes) // 2
    slopes_x   = slopes[:n_sub]           # x-slopes for all subapertures
    slopes_y   = slopes[n_sub:]           # y-slopes for all subapertures
    side       = int(np.sqrt(n_sub))      # grid side length

    # Reshape to 2D grid for visualisation
    slopes_x_2d = slopes_x.reshape(side, side)
    slopes_y_2d = slopes_y.reshape(side, side)

    # Plot x-slopes
    vmax_x = np.abs(slopes_x_2d).max() + 1e-10
    im_x   = axes[0, col].imshow(slopes_x_2d, cmap='RdBu_r',
                                   vmin=-vmax_x, vmax=vmax_x)
    axes[0, col].set_title(
        f'X-slopes — r₀={r0*100:.0f}cm\n'
        f'∂φ/∂x at each subaperture\n'
        f'std = {slopes_x.std():.4f} rad',
        fontsize=10
    )
    plt.colorbar(im_x, ax=axes[0, col], label='Slope (rad/m)', fraction=0.046)

    # Plot y-slopes
    vmax_y = np.abs(slopes_y_2d).max() + 1e-10
    im_y   = axes[1, col].imshow(slopes_y_2d, cmap='RdBu_r',
                                   vmin=-vmax_y, vmax=vmax_y)
    axes[1, col].set_title(
        f'Y-slopes — r₀={r0*100:.0f}cm\n'
        f'∂φ/∂y at each subaperture\n'
        f'std = {slopes_y.std():.4f} rad',
        fontsize=10
    )
    plt.colorbar(im_y, ax=axes[1, col], label='Slope (rad/m)', fraction=0.046)

    # Print slope statistics
    print(f"r₀={r0*100:.0f}cm | "
          f"slopes shape: ({len(slopes)},) = ({n_sub}×2) | "
          f"sx_std={slopes_x.std():.4f} | "
          f"sy_std={slopes_y.std():.4f}")

plt.tight_layout()
save_path = f'{DAY1_DIR}/day1_slope_vectors.png'
plt.savefig(save_path, dpi=200, bbox_inches='tight')
plt.show()

print(f"\n Saved: {save_path}")
print()
print(" Key observations:")
print("  r₀=20cm: slopes vary smoothly → easy to integrate back to phase")
print("  r₀=3cm : slopes vary rapidly → integration fails = branch points")
print("  Slope std INCREASES as r₀ decreases (more turbulence = bigger slopes)")
print()
print(" In Day 3 you will implement reconstruction from these slopes.")
print("In Day 4 your CNN will use BOTH slopes + spot image as dual inputs.")

In [ ]:
# ============================================================
# CELL 9: 20-Frame Time-Series Simulation
# Physics: wind moves turbulence → wavefront changes over time
# This is what ISRO's BMP file sequence looks like temporally
# ============================================================

print("Simulating 20-frame time-series at r₀=10cm (moderate turbulence)...")
print("Physics: wind at 10 m/s moves turbulence across the aperture")
print()

# Parameters for time-series
N_FRAMES    = 20          # Number of frames
DT          = 0.002       # 2ms between frames = 500Hz camera (ISRO: "few milliseconds")
R0_SERIES   = 0.10        # Fixed r₀ for this demo
WIND_SPEED  = 10.0        # m/s

# Build the atmosphere
pupil_grid_ts  = hcipy.make_pupil_grid(GRID_SIZE, D)
aperture_ts    = hcipy.circular_aperture(D)(pupil_grid_ts)
Cn2_ts         = hcipy.Cn_squared_from_fried_parameter(R0_SERIES, WAVELENGTH)

# SAME layer evolves over time — wind blows the frozen screen
# Physics: Taylor's frozen turbulence hypothesis
#   The atmosphere moves as a rigid screen blown by wind
#   This is why you can estimate wind speed from temporal correlations!
layer_ts = hcipy.InfiniteAtmosphericLayer(
    pupil_grid_ts, Cn2_ts, 100, [WIND_SPEED, 0], 1
)

# Storage arrays
frames_phases = []    # Phase maps over time: list of (H,W) arrays
frames_spots  = []    # Spot images over time: list of (H,W) arrays
frames_slopes = []    # Slope vectors over time: list of (2×N_sub,) arrays
frame_times   = []    # Timestamps in seconds
frame_rms     = []    # RMS wavefront error per frame

for frame_idx in range(N_FRAMES):
    # Evolve the atmosphere to the current time
    # Physics: wind moves the frozen turbulence screen by v×dt each step
    t_current = frame_idx * DT
    layer_ts.evolve_until(t_current)

    # Propagate flat wavefront through evolved atmosphere
    wf_flat = hcipy.Wavefront(aperture_ts, WAVELENGTH)
    wf_turb = layer_ts.forward(wf_flat)

    # Extract phase
    phase_rad  = wf_turb.phase.shaped
    phase_nm   = phase_rad * (WAVELENGTH * 1e9) / (2 * np.pi)
    mask       = aperture_ts.shaped > 0.5
    phase_nm[~mask] = np.nan
    rms        = np.sqrt(np.nanmean(phase_nm**2))

    # Generate spot image
    spot_img, slopes, _, _ = simulate_shwfs_frame(
        wf_turb, pupil_grid_ts, aperture_ts,
        n_lenslets=N_LENSLETS, mla_focal_length=MLA_FOCAL_LENGTH, D=D
    )

    frames_phases.append(phase_nm)
    frames_spots.append(spot_img)
    frames_slopes.append(np.array(slopes).flatten())
    frame_times.append(t_current)
    frame_rms.append(rms)

    if frame_idx % 5 == 0:
        print(f"  Frame {frame_idx+1:2d}/{N_FRAMES} | "
              f"t={t_current*1000:.1f}ms | "
              f"RMS={rms:.1f}nm")

# Convert to arrays
frames_phases = np.array(frames_phases)   # (20, 128, 128)
frames_spots  = np.array(frames_spots)    # (20, H, W)
frames_slopes = np.array(frames_slopes)   # (20, 2×N_sub)
frame_times   = np.array(frame_times)
frame_rms     = np.array(frame_rms)

print()
print(f" Time-series generated:")
print(f"   frames_phases shape : {frames_phases.shape}")
print(f"   frames_spots shape  : {frames_spots.shape}")
print(f"   frames_slopes shape : {frames_slopes.shape}")
print(f"   Time span           : 0 to {frame_times[-1]*1000:.1f}ms")
print(f"   Frame interval      : {DT*1000:.1f}ms ({1/DT:.0f}Hz)")

In [ ]:
# ============================================================
# CELL 10: Visualise Time-Series
# Shows: how wavefront + spots evolve over 20 frames
# ============================================================

# Plot 1: Selected frames from time-series
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle(
    f'SH-WFS Time-Series — r₀={R0_SERIES*100:.0f}cm, Wind={WIND_SPEED}m/s, Δt={DT*1000:.0f}ms\n'
    'Top: Spot images (BMP frames) | Bottom: True wavefront phases (ground truth)',
    fontsize=12, fontweight='bold'
)

selected_frames = [0, 4, 9, 14, 19]  # Frames to display

for col, fidx in enumerate(selected_frames):
    # Spot image
    axes[0, col].imshow(frames_spots[fidx], cmap='hot', origin='lower')
    axes[0, col].set_title(f't={frame_times[fidx]*1000:.0f}ms\nFrame {fidx+1}', fontsize=10)
    axes[0, col].axis('off')

    # Phase map
    vmax = np.nanpercentile(np.abs(frames_phases[fidx]), 98)
    axes[1, col].imshow(frames_phases[fidx], cmap='RdBu_r',
                         vmin=-vmax, vmax=vmax, origin='lower')
    axes[1, col].set_title(f'RMS={frame_rms[fidx]:.0f}nm', fontsize=10)
    axes[1, col].axis('off')

plt.tight_layout()
save_path = f'{DAY1_DIR}/day1_timeseries_frames.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Saved: {save_path}")

# Plot 2: RMS wavefront error over time
fig2, ax2 = plt.subplots(figsize=(12, 4))
ax2.plot(frame_times * 1000, frame_rms, 'b-o', linewidth=2,
          markersize=6, label='RMS wavefront error (nm)')
ax2.fill_between(frame_times * 1000, frame_rms,
                   alpha=0.2, color='blue')
ax2.axhline(np.mean(frame_rms), color='red', linestyle='--',
             label=f'Mean RMS = {np.mean(frame_rms):.1f}nm')
ax2.set_xlabel('Time (ms)', fontsize=12)
ax2.set_ylabel('RMS Wavefront Error (nm)', fontsize=12)
ax2.set_title(
    f'RMS Error Over Time — r₀={R0_SERIES*100:.0f}cm, Wind={WIND_SPEED}m/s\n'
    'This time-series is what M3\'s LSTM will learn to predict',
    fontsize=11
)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

# Annotate with τ₀
tau0_estimate = 0.314 * R0_SERIES / WIND_SPEED * 1000  # in ms
ax2.axvspan(0, tau0_estimate, alpha=0.1, color='green',
             label=f'τ₀ ≈ {tau0_estimate:.1f}ms')
ax2.text(tau0_estimate/2, np.max(frame_rms)*0.95,
          f'τ₀ ≈ {tau0_estimate:.1f}ms\n(coherence window)',
          ha='center', va='top', fontsize=9, color='darkgreen',
          bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
ax2.legend(fontsize=10)

plt.tight_layout()
save_path2 = f'{DAY1_DIR}/day1_rms_timeseries.png'
plt.savefig(save_path2, dpi=150, bbox_inches='tight')
plt.show()

print(f" Saved: {save_path2}")
print()
print(f" τ₀ estimate = 0.314 × r₀/v_wind = 0.314 × {R0_SERIES}/{WIND_SPEED} = {tau0_estimate:.1f}ms")
print(f"   This means the wavefront stays 'similar' for ~{tau0_estimate:.1f}ms")
print(f"   Your AO loop must run faster than {tau0_estimate:.1f}ms!")
print(f"   In Day 3 you will compute τ₀ from the data itself (no formula needed)")

In [ ]:
# ============================================================
# CELL 11: BMP File Pipeline
# ISRO says: "A time series of SH-WFS frames (essentially .bmp files)"
# Build: save → load → preprocess pipeline
# ============================================================

BMP_DIR = f'{DAY1_DIR}/bmp_frames'
os.makedirs(BMP_DIR, exist_ok=True)

print("Building BMP file pipeline...")
print("(Simulating what ISRO's camera software produces)")
print()

# ── Step 1: Save all frames as BMP files ──
print("Step 1: Saving frames as BMP files...")
saved_paths = []
for i, spot_frame in enumerate(frames_spots):
    bmp_path = f'{BMP_DIR}/frame_{i:04d}.bmp'
    # OpenCV writes BMP — C++ backend, very fast
    success  = cv2.imwrite(bmp_path, spot_frame)
    saved_paths.append(bmp_path)
    if i % 5 == 0:
        print(f"  Saved: frame_{i:04d}.bmp "
              f"| Size: {os.path.getsize(bmp_path)/1024:.1f}KB")

print(f"  Total: {len(saved_paths)} BMP files saved")
print()


# ── Step 2: Load BMP files and build sequence ──
def load_shwfs_sequence(bmp_dir, frame_dt_ms=2.0,
                         pixel_size_um=10.0,
                         n_lenslets=10):
    """
    Load a sequence of SH-WFS BMP frames into memory.

    This is the FIRST function your algorithm runs on ISRO's real data.
    It accepts all the parameters ISRO provides in their 'Data Required' section.

    Args:
        bmp_dir        : Path to folder containing BMP files
        frame_dt_ms    : Time between frames in milliseconds
        pixel_size_um  : Camera pixel size in micrometres
        n_lenslets     : Number of lenslets per side in MLA

    Returns:
        frames         : (N, H, W) numpy array of all frames
        timestamps_ms  : (N,) array of frame timestamps in ms
        metadata       : dict with calibration info
    """
    # Find all BMP files sorted by name
    bmp_files = sorted([
        f for f in os.listdir(bmp_dir)
        if f.lower().endswith('.bmp')
    ])

    if len(bmp_files) == 0:
        raise ValueError(f"No BMP files found in {bmp_dir}")

    frames_list = []
    for fname in bmp_files:
        path  = os.path.join(bmp_dir, fname)
        # cv2.imread: C++ backend — this is where the C-speed comes from
        frame = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if frame is None:
            print(f"  Warning: Could not read {fname}")
            continue
        frames_list.append(frame.astype(np.float32))

    frames       = np.array(frames_list)
    N            = len(frames)
    timestamps   = np.arange(N) * frame_dt_ms

    # Metadata dict — stores all calibration info
    metadata = {
        'n_frames'          : N,
        'frame_shape'       : frames[0].shape,
        'frame_dt_ms'       : frame_dt_ms,
        'pixel_size_um'     : pixel_size_um,
        'n_lenslets'        : n_lenslets,
        'n_subapertures'    : n_lenslets ** 2,
        'total_duration_ms' : timestamps[-1],
        'framerate_hz'      : 1000.0 / frame_dt_ms,
    }

    return frames, timestamps, metadata


print("Step 2: Loading BMP sequence...")
loaded_frames, timestamps_ms, meta = load_shwfs_sequence(
    bmp_dir      = BMP_DIR,
    frame_dt_ms  = DT * 1000,
    pixel_size_um= PIXEL_SIZE * 1e6,
    n_lenslets   = N_LENSLETS
)

print(f"   Loaded {meta['n_frames']} frames")
print(f"  Frame shape    : {meta['frame_shape']}")
print(f"  Frame interval : {meta['frame_dt_ms']}ms ({meta['framerate_hz']:.0f}Hz)")
print(f"  Total duration : {meta['total_duration_ms']:.1f}ms")
print(f"  Subapertures   : {meta['n_subapertures']}")
print()


# ── Step 3: Verify round-trip (save → load → same pixels) ──
print("Step 3: Verifying BMP round-trip accuracy...")
for i in range(3):
    original = frames_spots[i].astype(np.float32)
    reloaded = loaded_frames[i]
    max_diff = np.abs(original - reloaded).max()
    print(f"  Frame {i}: max pixel difference = {max_diff:.1f} "
          f"({' PASS' if max_diff < 1 else ' FAIL'})")

print()
print(" Pipeline is ready to process ISRO's real BMP data")
print("   When ISRO provides their BMP files, just change BMP_DIR")
print("   and this entire pipeline runs on their real data automatically.")

In [ ]:
# ============================================================
# CELL 12: Save All Day 1 Data to Drive
# Every future day's notebook loads from these files
# ============================================================

print("Saving Day 1 simulation data to Google Drive...")
print()

# Save wavefront phase maps + spot images per turbulence level
for label, r0 in R0_VALUES.items():
    save_file = f'{DAY1_DIR}/wavefront_{label}_r0_{int(r0*100)}cm.npz'
    np.savez(
        save_file,
        phase_nm  = wavefronts[label]['phase_nm'],
        spot_image= spot_data[label]['spot_image'],
        slopes    = np.array(spot_data[label]['slopes']).flatten(),
        r0        = np.array(r0),
        rms_nm    = np.array(wavefronts[label]['rms_nm']),
    )
    size_kb = os.path.getsize(save_file) / 1024
    print(f"  ✅ Saved {label} data: {os.path.basename(save_file)} ({size_kb:.1f}KB)")

# Save the 20-frame time-series
ts_save_file = f'{DAY1_DIR}/timeseries_r0_10cm_20frames.npz'
np.savez(
    ts_save_file,
    frames_phases = frames_phases,
    frames_spots  = frames_spots,
    frames_slopes = frames_slopes,
    frame_times   = frame_times,
    frame_rms     = frame_rms,
    r0            = np.array(R0_SERIES),
    dt_seconds    = np.array(DT),
    wind_speed    = np.array(WIND_SPEED),
)
size_kb = os.path.getsize(ts_save_file) / 1024
print(f"  ✅ Saved time-series: {os.path.basename(ts_save_file)} ({size_kb:.1f}KB)")

# Save simulation parameters
params_file = f'{DAY1_DIR}/simulation_params.npz'
np.savez(
    params_file,
    D              = np.array(D),
    wavelength     = np.array(WAVELENGTH),
    n_lenslets     = np.array(N_LENSLETS),
    mla_focal_length= np.array(MLA_FOCAL_LENGTH),
    pixel_size     = np.array(PIXEL_SIZE),
    grid_size      = np.array(GRID_SIZE),
)
print(f"   Saved parameters: {os.path.basename(params_file)}")

print()
print("=" * 55)
print("DAY 1 COMPLETE — Summary of deliverables")
print("=" * 55)
print(f" All files saved at: {DAY1_DIR}")
print()
print(" day1_wavefronts_comparison.png    → proposal Figure 1")
print(" day1_cnn_input_vs_target.png      → proposal Figure 2")
print(" day1_slope_vectors.png            → understanding slopes")
print(" day1_timeseries_frames.png        → time-series visualisation")
print(" day1_rms_timeseries.png           → τ₀ visualisation")
print(" bmp_frames/ folder               → 20 BMP files")
print(" wavefront_*.npz files             → training data foundation")
print(" timeseries_r0_10cm_20frames.npz   → LSTM foundation (Day 4)")
print()
print("=" * 55)
print("READY FOR DAY 1 TEAM SYNC")
print("=" * 55)
print()
print("Each member answers these before bed:")
print("  Q1: What is a wavefront phase map? (show the image you generated)")
print("  Q2: What is the CNN's input? (spot image)")
print("  Q3: What is the CNN's target? (phase map)")
print("  Q4: What is the slope vector shape for 10×10 MLA? (200,)")
print("  Q5: What is τ₀ for r₀=10cm, wind=10m/s? (~3.1ms)")
print("  Q6: What are the 5 outputs ISRO expects?")